In [ ]:
# Libraries
import numpy as np
import pandas as pd
import deap
import random

from collections import defaultdict
from deap import base, creator, tools

In [8]:
# Import prediction data
df = pd.read_csv("Results/RUL_predictions.csv")
RUL = dict(zip(df.engine_id, df.RUL_predicted))

In [9]:
# Various helper functions and other structures
def get_engine_due_date(engine_id, RUL):
    '''Computes the engine's safety due date (ts = t1 + Rj - 1), t1 = 1'''
    t1 = 1
    return t1 + RUL[engine_id] + 1

def get_mu_A(id):
    '''Returns the days to perform maintenance for team A'''
    if 1 <= id <= 20:
        return 5
    elif 21 <= id <= 55:
        return 3
    elif 56 <= id <= 80:
        return 4
    else:
        return 5
    
def get_maintenance_duration(engine_id, team_type):
    '''Returns the days to perform maintenance for the input team type and engine ID'''
    mu_A = get_mu_A(engine_id)

    if team_type == "A":
        return mu_A
    elif team_type == "B":
        if 1 <= engine_id <= 25:
            return mu_A - 1
        elif 26 <= engine_id <= 70:
            return mu_A + 3
        else:
            return mu_A + 2
    else:
        raise ValueError("get_maintenance_duration() received invalid team type!")
    
def get_cj_value(id):
    '''Returns the penalty value cj (c = cj * (t - ts)^2) for the engine id'''
    if 1 <= id <= 25:
        return 4
    elif 26 <= id <= 45:
        return 2
    elif 46 <= id <= 75:
        return 5
    else:
        return 6
    
def get_penalty_cost(engine_id, completion_date, due_date):
    '''Computes the penalty cost given the completion and due dates of an engine'''
    # In case completed in time, no penalty
    if completion_date <= due_date:
        return 0
    
    # Compute penalty value
    cj = get_cj_value(engine_id)
    delay = completion_date - due_date
    penalty = cj * (delay ** 2)
    return min(penalty, 250) # Return if less than 250, otherwise return 250

# Team type mapping
TEAM_TYPES = {
    1: "A",
    2: "B",
    3: "A",
    4: "B"
}

In [14]:
### MAIN GA CODE BLOCK ###

# Encoding the GA individual (chromosomes)
# General idea: [(engine, team, start_day), (engine, team, start_day), ...]
# So one gene is one assignment of maintenance; this structure should be flexible enough for all constraints and operations

# Build schedules
def build_schedule(individual):
    team_jobs = defaultdict(list)

    for engine, team, start in individual:
        team_jobs[team].append((engine, start))

    return team_jobs

def is_feasible(team_jobs):
    '''Returns whether or not a provided (individual's) schedule is feasible given the constraints'''
    for team, jobs in team_jobs.items():
        jobs = sorted(jobs, key=lambda x: x[1])
        current_time = 0

        for engine, start in jobs:
            duration = get_maintenance_duration(engine, TEAM_TYPES[team])
            if start < current_time: # If the start time of a job is before the current time, the schedule is obviously not feasible.
                return False
            if start + duration - 1 > 30: # Planning horizon of 30, so job duration cannot be longer than this.
                return False
            current_time = start + duration

    return True # If no issues caught during the loops, the schedule is valid.

def evaluate(individual, RUL):
    '''
    Calculates the total penalty for a given individual.
    Trailing commas in outputs are there for DEAP format.
    '''
    team_jobs = build_schedule(individual)

    if not is_feasible(team_jobs):
        return (1e9,) # Arbitrarily large penalty for invalid individuals
    
    maintained_engines = set()
    total_penalty = 0

    # Calculate penalties for maintained engines
    for team, jobs in team_jobs.items():
        for engine, start in jobs:
            duration = get_maintenance_duration(engine, TEAM_TYPES[team])
            completion = start + duration - 1
            due_date = get_engine_due_date(engine, RUL)
            total_penalty += get_penalty_cost(engine, completion, due_date)
            maintained_engines.add(engine)

    # Calculate penalties for non-maintained engines
    for engine in range(1, 101):
        if engine not in maintained_engines:
            due = get_engine_due_date(engine, RUL)
            total_penalty += get_penalty_cost(engine, 30, due)

    return (total_penalty,)

# DEAP Setup
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)
model = base.Toolbox()

# Generate individuals
def create_individual():
    n_jobs = random.randint(10, 40) # Ensure varied but reasonable schedule lengths
    indiv = []
    for i in range(n_jobs):
        engine = random.randint(1, 100)
        team = random.randint(1, 4)
        start = random.randint(1, 30)
        indiv.append((engine, team, start))

    return creator.Individual(indiv)

model.register("individual", create_individual)
model.register("population", tools.initRepeat, list, model.individual)

# Placeholder operators (TODO: better functions and run the model)
model.register("evaluate", evaluate, RUL=RUL)
model.register("mate", tools.cxTwoPoint)
model.register("mutate", tools.mutShuffleIndexes, indpb=0.2)
model.register("select", tools.selTournament, tournsize=3)